<a href="https://colab.research.google.com/github/isaiahdm792-alt/stock--screener/blob/main/01_full_scan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import requests
from io import StringIO # Import StringIO

def get_sp500_tickers():
    """Scrape the current S&P 500 constituent list from Wikipedia."""
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    response = requests.get(url, headers=headers)
    response.raise_for_status()  # Raise an HTTPError for bad responses (4xx or 5xx)

    # Wrap the response text in StringIO as suggested by the FutureWarning
    tables = pd.read_html(StringIO(response.text))
    sp500_table = tables[0]  # first table on the page is the constituent list
    tickers = sp500_table["Symbol"].tolist()
    # yfinance expects dots as hyphens for some tickers (e.g. BRK.B -> BRK-B)
    tickers = [t.replace(".", "-") for t in tickers]
    return tickers

sp500_tickers = get_sp500_tickers()
print(f"Pulled {len(sp500_tickers)} tickers.")
print(sp500_tickers[:10], "...")

Pulled 503 tickers.
['MMM', 'AOS', 'ABT', 'ABBV', 'ACN', 'ADBE', 'AMD', 'AES', 'AFL', 'A'] ...


In [2]:
# STEP 1: Fundamentals Scanner — starter script
# Paste this into a Google Colab cell and run it.
# First run: install yfinance (only needed once per Colab session)

# !pip install yfinance --quiet

import yfinance as yf
import pandas as pd
import time

# --- Start small: 15 well-known tickers to test the pipeline first ---
# Once this works cleanly, swap in the full S&P 500 list (step below).
test_tickers = sp500_tickers

def get_fundamentals(ticker):
    """Pull key fundamental ratios for one ticker. Returns a dict or None on failure."""
    try:
        stock = yf.Ticker(ticker)
        info = stock.info

        pe = info.get("trailingPE")
        ps = info.get("priceToSalesTrailing12Months")
        pb = info.get("priceToBook")
        peg = info.get("pegRatio")
        fcf = info.get("freeCashflow")
        earnings_growth = info.get("earningsGrowth")
        sector = info.get("sector")

        return {
            "ticker": ticker,
            "sector": sector,
            "pe_ratio": pe,
            "ps_ratio": ps,
            "pb_ratio": pb,
            "peg_ratio": peg,
            "free_cash_flow": fcf,
            "earnings_growth": earnings_growth,
        }
    except Exception as e:
        print(f"  [!] Failed on {ticker}: {e}")
        return None

# --- Run the scan ---
results = []
for t in test_tickers:
    print(f"Pulling {t}...")
    data = get_fundamentals(t)
    if data:
        results.append(data)
    time.sleep(0.5)  # be polite to the API, avoid rate-limit issues

df = pd.DataFrame(results)

# --- Basic cleanup: drop rows missing the core ratios ---
df_clean = df.dropna(subset=["pe_ratio", "ps_ratio"])

print("\n--- Results ---")
print(df_clean.sort_values("pe_ratio").to_string(index=False))

# --- Save to CSV so you can commit it to GitHub / inspect later ---
df_clean.to_csv("fundamentals_snapshot.csv", index=False)
print("\nSaved to fundamentals_snapshot.csv")

Pulling MMM...
Pulling AOS...
Pulling ABT...
Pulling ABBV...
Pulling ACN...
Pulling ADBE...
Pulling AMD...
Pulling AES...
Pulling AFL...
Pulling A...
Pulling APD...
Pulling ABNB...
Pulling AKAM...
Pulling ALB...
Pulling ARE...
Pulling ALGN...
Pulling ALLE...
Pulling LNT...
Pulling ALL...
Pulling GOOGL...
Pulling GOOG...
Pulling MO...
Pulling AMZN...
Pulling AMCR...
Pulling AEE...
Pulling AEP...
Pulling AXP...
Pulling AIG...
Pulling AMT...
Pulling AWK...
Pulling AMP...
Pulling AME...
Pulling AMGN...
Pulling APH...
Pulling ADI...
Pulling AON...
Pulling APA...
Pulling APO...
Pulling AAPL...
Pulling AMAT...
Pulling APP...
Pulling APTV...
Pulling ACGL...
Pulling ADM...
Pulling ARES...
Pulling ANET...
Pulling AJG...
Pulling AIZ...
Pulling T...
Pulling ATO...
Pulling ADSK...
Pulling ADP...
Pulling AZO...
Pulling AVB...
Pulling AVY...
Pulling AXON...
Pulling BKR...
Pulling BALL...
Pulling BAC...
Pulling BAX...
Pulling BDX...
Pulling BRK-B...
Pulling BBY...
Pulling TECH...
Pulling BIIB...
Pulli

In [3]:
# STEP 2: Basic sector-relative scoring
import pandas as pd

df = df_clean.copy()  # use the results from Step 1

# Calculate the average P/E per sector
sector_avg_pe = df.groupby("sector")["pe_ratio"].transform("mean")

# Flag stocks trading below their sector average P/E
df["pe_vs_sector"] = df["pe_ratio"] - sector_avg_pe
df["undervalued_flag"] = df["pe_vs_sector"] < 0

# Simple composite score: lower PEG and lower relative P/E = higher score
df["score"] = (
    (1 / df["peg_ratio"].clip(lower=0.1)) * 40   # reward low PEG
    - df["pe_vs_sector"].clip(lower=0) * 0.5      # penalize high relative P/E
)

df_ranked = df.sort_values("score", ascending=False)

print(df_ranked[["ticker", "sector", "pe_ratio", "peg_ratio", "undervalued_flag", "score"]].head(20).to_string(index=False))

df_ranked.to_csv("scored_snapshot.csv", index=False)
print("\nSaved to scored_snapshot.csv")

ticker             sector   pe_ratio  peg_ratio  undervalued_flag      score
   HIG Financial Services   9.810083       0.12              True 333.333333
  CSGP        Real Estate 158.611100       0.11             False 311.140663
    MU         Technology  19.536802       0.13              True 307.692308
   DAL        Industrials  14.589170       0.21              True 190.476190
   COF Financial Services  11.420155       0.22              True 181.818182
   FIS         Technology   8.806202       0.24              True 166.666667
   GPN        Industrials  31.497242       0.24              True 166.666667
    SW  Consumer Cyclical  50.164894       0.26             False 150.202756
    ON         Technology  62.136030       0.29              True 137.931034
   TPR  Consumer Cyclical  46.075990       0.30             False 131.734388
   CVS         Healthcare  46.013160       0.30             False 126.482047
    GM  Consumer Cyclical  38.799110       0.35              True 114.285714

In [5]:
# STEP 2: Technical Indicators — no extra libraries needed, avoids pandas-ta version conflicts
# Run this after your fundamentals scan (Part 1) is done and df_ranked exists.
# No pip installs needed for this version - just yfinance and pandas, which you already have.

import yfinance as yf
import pandas as pd
import time

def calculate_rsi(prices, period=14):
    """Manual RSI calculation - no external library needed."""
    delta = prices.diff()
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)
    avg_gain = gain.rolling(window=period).mean()
    avg_loss = loss.rolling(window=period).mean()
    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

def calculate_macd(prices, fast=12, slow=26, signal=9):
    """Manual MACD calculation - no external library needed."""
    ema_fast = prices.ewm(span=fast, adjust=False).mean()
    ema_slow = prices.ewm(span=slow, adjust=False).mean()
    macd_line = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal, adjust=False).mean()
    return macd_line, signal_line

def get_technicals(ticker):
    """Pull price history and calculate SMA, RSI, MACD, and volume anomaly for one ticker."""
    try:
        hist = yf.Ticker(ticker).history(period="1y")
        if hist.empty or len(hist) < 200:
            return None  # not enough history to calculate 200-day SMA

        hist["sma50"] = hist["Close"].rolling(window=50).mean()
        hist["sma200"] = hist["Close"].rolling(window=200).mean()
        hist["rsi"] = calculate_rsi(hist["Close"])
        hist["macd"], hist["macd_signal"] = calculate_macd(hist["Close"])

        avg_volume_30d = hist["Volume"].tail(30).mean()
        latest = hist.iloc[-1]

        return {
            "ticker": ticker,
            "price": latest["Close"],
            "sma50": latest["sma50"],
            "sma200": latest["sma200"],
            "above_sma50": latest["Close"] > latest["sma50"],
            "above_sma200": latest["Close"] > latest["sma200"],
            "rsi": latest["rsi"],
            "macd_bullish": latest["macd"] > latest["macd_signal"],
            "volume_today": latest["Volume"],
            "avg_volume_30d": avg_volume_30d,
            "volume_spike": latest["Volume"] > (avg_volume_30d * 1.5),
        }
    except Exception as e:
        print(f"  [!] Failed on {ticker}: {e}")
        return None

# --- Test on a small slice first, same pattern as Step 1 ---
test_slice = sp500_tickers

tech_results = []
for t in test_slice:
    print(f"Pulling technicals for {t}...")
    data = get_technicals(t)
    if data:
        tech_results.append(data)
    time.sleep(0.5)

df_tech = pd.DataFrame(tech_results)
print("\n--- Technical Results ---")
print(df_tech.to_string(index=False))

df_tech.to_csv("technicals_snapshot.csv", index=False)
print("\nSaved to technicals_snapshot.csv")

Pulling technicals for MMM...
Pulling technicals for AOS...
Pulling technicals for ABT...
Pulling technicals for ABBV...
Pulling technicals for ACN...
Pulling technicals for ADBE...
Pulling technicals for AMD...
Pulling technicals for AES...
Pulling technicals for AFL...
Pulling technicals for A...
Pulling technicals for APD...
Pulling technicals for ABNB...
Pulling technicals for AKAM...
Pulling technicals for ALB...
Pulling technicals for ARE...
Pulling technicals for ALGN...
Pulling technicals for ALLE...
Pulling technicals for LNT...
Pulling technicals for ALL...
Pulling technicals for GOOGL...
Pulling technicals for GOOG...
Pulling technicals for MO...
Pulling technicals for AMZN...
Pulling technicals for AMCR...
Pulling technicals for AEE...
Pulling technicals for AEP...
Pulling technicals for AXP...
Pulling technicals for AIG...
Pulling technicals for AMT...
Pulling technicals for AWK...
Pulling technicals for AMP...
Pulling technicals for AME...
Pulling technicals for AMGN...
P

In [7]:
# Normalize each sub-score to 0-100 before combining
def normalize(series):
    return (series - series.min()) / (series.max() - series.min()) * 100

df_combined["fundamentals_score"] = normalize(1 / df_combined["peg_ratio"].clip(lower=0.1))
df_combined["technicals_score"] = (
    df_combined["above_sma50"].astype(int) * 25
    + df_combined["above_sma200"].astype(int) * 25
    + df_combined["macd_bullish"].astype(int) * 25
    + df_combined["volume_spike"].astype(int) * 25
)  # already naturally 0-100

# Now combine with deliberate weights (adjust these as you like)
df_combined["score"] = (
    df_combined["fundamentals_score"] * 0.6
    + df_combined["technicals_score"] * 0.4
)

df_combined = df_combined.sort_values("score", ascending=False)
print(df_combined[["ticker", "fundamentals_score", "technicals_score", "score"]].head(20).to_string(index=False))

df_combined.to_csv("scored_snapshot.csv", index=False)

ticker  fundamentals_score  technicals_score     score
   HIG           91.660251                75 84.996150
  CSGP          100.000000                25 70.000000
    MU           84.603540                25 60.762124
   GPN           45.791629                75 57.474977
    SW           42.263273                75 55.357964
   TPR           36.617904                75 51.970743
   DAL           52.344289                50 51.406573
   COF           49.961503                50 49.976902
    GM           31.375776                75 48.825466
   FIS           45.791629                50 47.474977
  MSFT            9.020915               100 45.412549
   MET           21.939945                75 43.163967
   CVS           36.617904                50 41.970743
  ZBRA           19.580988                75 41.748593
  AMCR           17.678602                75 40.607161
   ZBH           16.859113                75 40.115468
   APA           16.602506                75 39.961503
   CVX    